# NextBuy — reproducible grocery intelligence

This notebook uses the same validated modules as the dashboard. The original academic notebook remains in `nextbuy/notebook.ipynb` as historical evidence; its saved conclusions are not the validation record for this restoration.

**Task:** exploratory basket analytics, exact co-purchase pairs and diagnostic classification of repeat status among purchased articles. No future purchase, revenue or promotion uplift is predicted.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "nextbuy_core").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from nextbuy_core.data import available_sources, load_tables, merge_tables, quality_report
from nextbuy_core.analytics import kpis, product_statistics, order_activity, association_pairs
from nextbuy_core.modeling import run_experiment
import plotly.express as px
sources = available_sources()
source = next(iter(sources))
tables = load_tables(sources[source])
frame = merge_tables(tables)
print("ACTIVE SOURCE:", source)
print("Synthetic data illustrate the software only." if "Synthetic" in source else "Local Instacart cohort; partial source extract.")
display(quality_report(tables))
display(kpis(frame))

ACTIVE SOURCE: Instacart · local cohort
Local Instacart cohort; partial source extract.


,table,rows,columns,null_cells,duplicate_keys
0,orders,20256,6,1272,0
1,products,18951,4,0,0
2,aisles,134,2,0,0
3,departments,21,2,0,0
4,order_products,202199,4,0,0


{'orders': 20256,
 'customers': 2948,
 'items': 202199,
 'repeat_rate': 0.5886774909866023,
 'basket_size': 9.98217812006319}

## Exploration
Hours count distinct orders. Product counts measure article lines, not units or revenue. First-order elapsed intervals stay missing. No weekday mapping or calendar period is assumed.

In [2]:
display(product_statistics(frame).nlargest(15, "items"))
display(product_statistics(frame, "department").sort_values("items", ascending=False))
hourly = order_activity(frame, "order_hour_of_day")
px.line(hourly, x="order_hour_of_day", y="orders", title="Distinct orders by hour").show()
baskets = frame.groupby("order_id").size().rename("articles").reset_index()
px.histogram(baskets, x="articles", title="Observed basket sizes — all sizes retained").show()

,product_name,items,repeat_rate,first_count,first_ratio
1295,Banana,3038,0.849901,692,0.227781
1210,Bag of Organic Bananas,2298,0.820714,468,0.203655
12277,Organic Strawberries,1680,0.767262,197,0.117262
10576,Organic Baby Spinach,1586,0.778689,149,0.093947
11383,Organic Hass Avocado,1421,0.806474,145,0.102041
10554,Organic Avocado,1160,0.771552,149,0.128448
8205,Large Lemon,1035,0.693720,81,0.078261
16444,Strawberries,919,0.694233,120,0.130577
12580,Organic Whole Milk,894,0.857942,213,0.238255
11958,Organic Raspberries,881,0.760499,89,0.101022


,department,items,repeat_rate,first_count,first_ratio
19,produce,59357,0.648197,5664,0.095423
7,dairy eggs,34089,0.671155,4124,0.120977
20,snacks,17942,0.566938,1393,0.077639
3,beverages,16896,0.654652,2626,0.155421
10,frozen,14339,0.545296,1225,0.085431
16,pantry,11616,0.347624,965,0.083075
2,bakery,7123,0.623333,699,0.098133
6,canned goods,6704,0.466140,380,0.056683
8,deli,6245,0.588311,543,0.086950
9,dry goods pasta,5405,0.461425,356,0.065865


## Associations
Exact enumeration of pairs among the 30 most frequent products, on complete observed baskets. Support divides by all observed orders, including orders without those products. Count thresholds reduce rare-pair instability. Lift is association, not a causal promotion effect.

In [3]:
pairs = association_pairs(frame, top_n=30, min_count=5)
display(pairs.head(20))
first = product_statistics(frame)
display(first.loc[first["items"].ge(20)].nlargest(10, "first_ratio"))

,product_a,product_b,count,support,confidence_a_to_b,lift
0,Bag of Organic Bananas,Organic Hass Avocado,405,0.019994,0.176240,2.512260
1,Bag of Organic Bananas,Organic Strawberries,368,0.018167,0.160139,1.930822
2,Organic Strawberries,Banana,348,0.017180,0.207143,1.381134
3,Banana,Organic Avocado,335,0.016538,0.110270,1.925541
4,Bag of Organic Bananas,Organic Baby Spinach,324,0.015995,0.140992,1.800717
5,Organic Baby Spinach,Banana,323,0.015946,0.203657,1.357892
6,Banana,Large Lemon,307,0.015156,0.101053,1.977716
7,Organic Strawberries,Organic Hass Avocado,279,0.013774,0.166071,2.367307
8,Banana,Organic Fuji Apple,270,0.013329,0.088874,2.667018
9,Bag of Organic Bananas,Organic Raspberries,254,0.012539,0.110531,2.541332


,product_name,items,repeat_rate,first_count,first_ratio
278,2-Ply Right Size 100% Recycled Paper Towels,54,0.796296,30,0.555556
18265,Vodka,48,0.770833,26,0.541667
15791,Smartwater,63,0.825397,33,0.523810
11589,Organic Low Fat Milk,40,0.675000,20,0.500000
11675,"Organic Milk Reduced Fat, 2% Milkfat",49,0.755102,23,0.469388
784,Alpine Spring Water,33,0.545455,15,0.454545
12491,Organic Variety Pack,31,0.806452,14,0.451613
6413,Gluten Free Matzo-Style Squares,23,0.913043,10,0.434783
17414,Traditional Scottish Style Smoked Salmon,23,0.739130,10,0.434783
14263,Purified Water,72,0.805556,31,0.430556


## Fixed model evaluation
Older observed baskets supply history. Penultimate baskets supply training labels. Last observed baskets supply test labels. Customers need at least three observed orders. Smoothed product/user rates use only the separate history partition and stay frozen. All estimators use independent preprocessing pipelines. Numeric predictors are standardised; unknown categories are accepted.

The constant prevalence classifier, logistic regression and random forest use fixed settings and seed 42. No test-driven hyperparameter or winner selection. Cart position is post-selection information: this is retrospective classification, not a recommendation engine or next-purchase model.

In [4]:
result = run_experiment(frame)
display(result["sizes"])
display(result["scores"])
for name, detail in result["details"].items():
    print(name)
    display(detail["report"])
    display(detail["confusion"])

{'history_items': 145303,
 'train_items': 21010,
 'test_items': 21729,
 'excluded_users': 890}

,model,ROC_AUC,Average_precision,Brier_score
0,Prevalence baseline,0.500000,0.639054,0.231467
1,Logistic regression,0.737694,0.825968,0.196577
2,Random forest,0.748398,0.836000,0.194595


Prevalence baseline


{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 7843.0},
 '1': {'precision': 0.6390537990703667,
  'recall': 1.0,
  'f1-score': 0.7797837989611119,
  'support': 13886.0},
 'accuracy': 0.6390537990703667,
 'macro avg': {'precision': 0.3195268995351834,
  'recall': 0.5,
  'f1-score': 0.38989189948055597,
  'support': 21729.0},
 'weighted avg': {'precision': 0.40838975810626876,
  'recall': 0.6390537990703667,
  'f1-score': 0.49832379917962166,
  'support': 21729.0}}

array([[    0,  7843],
       [    0, 13886]])

...

{'0': {'precision': 0.5890192669853687,
  'recall': 0.5184240724212674,
  'f1-score': 0.5514715855147159,
  'support': 7843.0},
 '1': {'precision': 0.74524484014569,
  'recall': 0.7956935042488837,
  'f1-score': 0.7696433546949011,
  'support': 13886.0},
 'accuracy': 0.6956141561967877,
 'macro avg': {'precision': 0.6671320535655294,
  'recall': 0.6570587883350756,
  'f1-score': 0.6605574701048085,
  'support': 21729.0},
 'weighted avg': {'precision': 0.6888558130254175,
  'recall': 0.6956141561967877,
  'f1-score': 0.6908950834592165,
  'support': 21729.0}}

array([[ 4066,  3777],
       [ 2837, 11049]])

Random forest


{'0': {'precision': 0.6495967741935483,
  'recall': 0.41081218921331125,
  'f1-score': 0.5033195344841053,
  'support': 7843.0},
 '1': {'precision': 0.7244319875961596,
  'recall': 0.8748379662969897,
  'f1-score': 0.7925623878649486,
  'support': 13886.0},
 'accuracy': 0.7073496249252151,
 'macro avg': {'precision': 0.6870143808948539,
  'recall': 0.6428250777551505,
  'f1-score': 0.6479409611745269,
  'support': 21729.0},
 'weighted avg': {'precision': 0.6974205016227286,
  'recall': 0.7073496249252151,
  'f1-score': 0.6881612787910864,
  'support': 21729.0}}

array([[ 3222,  4621],
       [ 1738, 12148]])

## Limitations
The original extract is incomplete and contains corrupt rows; preprocessing excludes entire affected baskets and records exclusions in the local manifest. This changes the analysis population. Last observed orders may not be the customer's actual last order. Ordering is relative within customer, not global chronological validation, because calendar dates are unavailable. Observations are dependent within customers; no confidence intervals, probability calibration, external validity or causal effects are established. Synthetic and real results must be reported separately.